In [57]:
import numpy as np
from sklearn.cluster import KMeans

# ============================================================
# EXERCISE 1.2 — ITEM DEFINITIONS
# ============================================================

# 5 items, 4-dimensional embeddings (toy example)
items = np.array([
    [1.0, 1.1, 4.9, 5.1],   # v1 -> index 0
    [1.2, 0.9, 5.0, 4.8],   # v2 -> index 1
    [0.9, 1.0, 4.7, 5.2],   # v3 -> index 2
    [8.0, 8.2, 1.1, 1.0],   # v4 -> index 3
    [7.8, 7.9, 1.0, 1.2]    # v5 -> index 4
])

item_names = ["v1","v2","v3","v4","v5"]

print("ITEM MATRIX:")
print(items)
print("\nItem names:", item_names)

# ============================================================
# EXERCISE 1.3 — AUTOMATIC IVF CLUSTERING (K-Means)
# ============================================================

# Run KMeans to cluster items into 2 coarse regions
kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(items)

print("\nCluster assignments (IVF coarse regions):")
for name, lbl in zip(item_names, labels):
    print(f"{name} -> cluster {lbl}")

# Extract IVF centroids
centroids = kmeans.cluster_centers_

centroid0 = centroids[0]
centroid1 = centroids[1]

print("\nIVF Centroids:")
print("Centroid 0:", centroid0)
print("Centroid 1:", centroid1)

# ============================================================
# EXERCISE 2.1 — SPLIT ITEMS INTO PQ BLOCKS
# ============================================================

# We split each 4-D item vector into 2 blocks of size 2 dims
num_blocks = 2
block_dim = 2

blocks = []
block1_data = []
block2_data = []

for i, vec in enumerate(items):
    b1 = vec[:2]    # dims [x1, x2]
    b2 = vec[2:]    # dims [x3, x4]
    blocks.append([b1, b2])
    block1_data.append(b1)
    block2_data.append(b2)

block1_data = np.array(block1_data)
block2_data = np.array(block2_data)

print("\nBLOCK 1 MATRIX:")
print(block1_data)
print("\nBLOCK 2 MATRIX:")
print(block2_data)

# ============================================================
# EXERCISE 2.2 — PQ CODEBOOK TRAINING (K-Means per block)
# ============================================================

# KMeans for Block 1
kmeans_block1 = KMeans(n_clusters=2, random_state=42)
labels_block1 = kmeans_block1.fit_predict(block1_data)
codebook_block1 = kmeans_block1.cluster_centers_

print("\nBlock 1 PQ codebook (centroids):")
print(codebook_block1)
print("Block 1 labels:", labels_block1)

kmeans_block2 = KMeans(n_clusters=2, random_state=42)
labels_block2 = kmeans_block2.fit_predict(block2_data)
codebook_block2 = kmeans_block2.cluster_centers_

print("\nBlock 2 PQ codebook (centroids):")
print(codebook_block2)
print("Block 2 labels:", labels_block2)

pq_codes = []

for i in range(len(items)):
    code_block1 = labels_block1[i]  # this item’s block-1 codeword index
    code_block2 = labels_block2[i]  # this item’s block-2 codeword index
    pq_codes.append([code_block1, code_block2])

pq_codes = np.array(pq_codes)

print("\nPQ codes for each item:")
for name, code in zip(item_names, pq_codes):
    print(f"{name}: PQ code = {code}")

# Show original vectors vs PQ codes
print("\nOriginal vectors vs PQ codes:")
for i, name in enumerate(item_names):
    print(f"{name} -> Vec: {items[i]},   PQ code: {pq_codes[i]}")

# ============================================================
# CONTINUATION: PQ RETRIEVAL (EXERCISES 3.1 → 3.4)
# We assume ALL variables from 2.3 already exist:
# items, item_names, pq_codes, codebook_block1, codebook_block2
# ============================================================

# USER VECTOR (4 dims)
u = np.array([1.0, 1.1, 4.8, 5.0])

print("User vector u:", u)

# ------------------------------------------------------------
# EXERCISE 3.1 — Split user into PQ blocks
# ------------------------------------------------------------

u_block1 = u[:2]     # dims 1–2
u_block2 = u[2:]     # dims 3–4

print("\nUser Block 1:", u_block1)
print("User Block 2:", u_block2)

# ------------------------------------------------------------
# EXERCISE 3.2 — Build PQ lookup tables (dot product)
# ------------------------------------------------------------

def build_lookup_table(u_block, codebook):

    return np.array([np.dot(u_block, cw) for cw in codebook])

LUT1 = build_lookup_table(u_block1, codebook_block1)
LUT2 = build_lookup_table(u_block2, codebook_block2)


print('LUT1', LUT1 )

print('LUT2', LUT2 )

# ============================================================
# EXERCISE 3.3 — Approximate similarity (dot product) using PQ codes
# ============================================================

# ============================================================
# FINAL EXERCISE — FULL SEARCH FUNCTION (IVF + PQ)
# ============================================================

def search(u, items, item_names, pq_codes, codebook_block1, codebook_block2, centroids, labels, top_k=3):
    """
    Performs ANN retrieval using IVF + PQ (dot-product).

    Steps:
    1. Split user into 2 blocks
    2. Compute dot-product similarity to IVF centroids → SELECT cluster
    3. Build PQ lookup tables for both blocks
    4. For all items in that IVF cluster:
         - Use pq_codes to compute approximate dot-product
    5. Return top-K items by similarity
    """

    # --------------------------------------------------------
    # Step 1 — Split user into PQ blocks
    # --------------------------------------------------------
    u_block1 = u[:2]
    u_block2 = u[2:]

    # --------------------------------------------------------
    # Step 2 — IVF cluster selection (dot-product)
    # --------------------------------------------------------
    sim0 = np.dot(u, centroids[0])
    sim1 = np.dot(u, centroids[1])
    chosen_cluster = 0 if sim0 > sim1 else 1

    # --------------------------------------------------------
    # Step 3 — PQ Lookup Tables
    # --------------------------------------------------------
    LUT1 = np.array([np.dot(u_block1, cw) for cw in codebook_block1])
    LUT2 = np.array([np.dot(u_block2, cw) for cw in codebook_block2])

    # --------------------------------------------------------
    # Step 4 — PQ approximate similarity for items in IVF cluster
    # --------------------------------------------------------
    results = []

    for i in range(len(items)):
        if labels[i] == chosen_cluster:
            c1, c2 = pq_codes[i]
            approx_sim = LUT1[c1] + LUT2[c2]
            results.append((item_names[i], approx_sim))

    # Sort by similarity (desc)
    results.sort(key=lambda x: x[1], reverse=True)

    # --------------------------------------------------------
    # Step 5 — Return top-K items
    # --------------------------------------------------------
    return results[:top_k]

    # Run search
results = search(u, items, item_names, pq_codes,
                 codebook_block1, codebook_block2,
                 centroids, labels, top_k=3)

print("\nTop-K ANN Recommendations for user u:")
for name, score in results:
    print(f"{name}: approx_similarity={score:.4f}")



ITEM MATRIX:
[[1.  1.1 4.9 5.1]
 [1.2 0.9 5.  4.8]
 [0.9 1.  4.7 5.2]
 [8.  8.2 1.1 1. ]
 [7.8 7.9 1.  1.2]]

Item names: ['v1', 'v2', 'v3', 'v4', 'v5']

Cluster assignments (IVF coarse regions):
v1 -> cluster 0
v2 -> cluster 0
v3 -> cluster 0
v4 -> cluster 1
v5 -> cluster 1

IVF Centroids:
Centroid 0: [1.03333333 1.         4.86666667 5.03333333]
Centroid 1: [7.9  8.05 1.05 1.1 ]

BLOCK 1 MATRIX:
[[1.  1.1]
 [1.2 0.9]
 [0.9 1. ]
 [8.  8.2]
 [7.8 7.9]]

BLOCK 2 MATRIX:
[[4.9 5.1]
 [5.  4.8]
 [4.7 5.2]
 [1.1 1. ]
 [1.  1.2]]

Block 1 PQ codebook (centroids):
[[1.03333333 1.        ]
 [7.9        8.05      ]]
Block 1 labels: [0 0 0 1 1]

Block 2 PQ codebook (centroids):
[[4.86666667 5.03333333]
 [1.05       1.1       ]]
Block 2 labels: [0 0 0 1 1]

PQ codes for each item:
v1: PQ code = [0 0]
v2: PQ code = [0 0]
v3: PQ code = [0 0]
v4: PQ code = [1 1]
v5: PQ code = [1 1]

Original vectors vs PQ codes:
v1 -> Vec: [1.  1.1 4.9 5.1],   PQ code: [0 0]
v2 -> Vec: [1.2 0.9 5.  4.8],   PQ code: 